# Fig 4h and Supp Fig 4c–d: neighborhood abundance and overall survival

**Produces**
- **Fig 4h**: RMST forest plot of the ECM neighborhoods, high vs low neighborhood fraction (median
  split), τ = 3 years, adjusted for MGMT, age, sex and p53; 207 patients (104 / 103). ECM_3
  −3.5 months [−6.2, −0.8], p = 0.0114; ECM_1 +1.6 months, p = 0.2595 (+2.7 months, p = 0.083 unadjusted).
- The † of Fig 4h: neighborhoods significant at τ = 1.5 years in the same model (ECM_3, −1.7 months, p = 0.013).
- **Supp Fig 4c–d**: the same sweep and forest plot on the cell-type neighborhoods: CT_3 −3.9 months,
  p = 0.006; CT_0 +3.3 months [0.4, 6.2], p = 0.026; and CT_3 / CT_0 adjusted for each other
  (CT_3 −3.0 months, p = 0.036; CT_0 +2.85 months, p = 0.052).

The sweep runs three splits (median, quartile, top quartile); p values are nominal. MGMT, sex
and p53 are coded by equality, so a missing value counts as unmethylated, male or negative; a
section links to its subject by removing `_vis` and then the last letter, so the 21 backup
sections find no subject (printed by `describe_fit`).

**Origin**: Spencer Watson, notebook Updated_figures (not in the repository), the RMST sweep under
"Updated ECM Neighborhood Plot" and the τ = 1.5 years check after it (as reproduced in
`scratch/review_2026-11/source_cells_spencer_2026-11.ipynb`, cells 6–9).

**Compared with the published panel**: Fig 4h reproduced (ECM_3 adjusted p = 0.0114, ECM_9 0.4743;
ECM_1 0.2595 against 0.2598 published). CT_3 and CT_0 match the numbers of the text.

In [ ]:
import functools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_hex

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import cohort, config, output

import rpy2.robjects as ro  # noqa: E402  (after config, which sets R_HOME)
from rpy2.robjects import pandas2ri  # noqa: E402
from rpy2.robjects.conversion import localconverter  # noqa: E402

ro.r("suppressPackageStartupMessages(library(survRM2))")

In [ ]:
# Parameters
OUT = output.setup("fig4h_suppfig4cd_neighborhood_survival")
ECM_CSV = config.ECM_ANNOTATIONS_CSV
CT_CSV = config.CT_ANNOTATIONS_CSV
TITLE = "{name} neighborhood survival associations (tau={tau}y, {split} split)\nCovariates: {covs}  |  colored = p<0.05, gray = n.s."

# Config
min_spots  = 10
tau_years  = 3
endpoint   = "OS"
cmap_name  = "tab20"

SPLIT_MODES = ["median", "quartile", "max"]

# Toggle covariates on/off
USE_MGMT = True
USE_AGE  = True
USE_SEX  = True
USE_P53  = True

FOREST_SPLIT = "median"

# Fig 4h dagger
TAU_CHECK = 1.5
ALPHA     = 0.05

# Aesthetics
FP_FIG_W     = 9
FP_FIG_H     = 6.0
FP_FS_TITLE  = 9
FP_FS_AXIS   = 12
FP_FS_HDR    = 8
FP_FS_ROW    = 8
FP_MARKER    = "D"
FP_MARKER_SZ = 5
FP_CAP_SZ    = 3
FP_LW        = 3
FP_ELW       = 2
FP_COL_NS    = "#888888"
FP_SIG_ALPHA = 0.05
X_LABEL      = -0.02
X_EST        =  1.02
X_P          =  1.28

# Covariate definitions
COVARIATE_DEFS = {
    "mgmt": {"col": "MGMT promoter methylation",
             "transform": lambda x: (x == "Methylated").astype(float), "active": USE_MGMT},
    "age":  {"col": "age_at_diagnosis",
             "transform": lambda x: x.astype(float), "active": USE_AGE},
    "sex":  {"col": "Administrative gender",
             "transform": lambda x: (x == "Female").astype(float), "active": USE_SEX},
    "p53":  {"col": "p53",
             "transform": lambda x: (x == "Positive").astype(float), "active": USE_P53},
}

COVARIATES = {k: v for k, v in COVARIATE_DEFS.items() if v["active"]}
cov_names  = list(COVARIATES.keys())
cov_label  = " + ".join(cov_names) if cov_names else "none"
adjust     = len(cov_names) > 0

# Endpoint
if endpoint.upper() == "OS":
    time_col, event_col = "os_years", "os_event"
    time_label = "Overall Survival"
else:
    time_col, event_col = "pfs_years", "pfs_event"
    time_label = "Progression-Free Survival"

In [ ]:
@functools.lru_cache
def clinical_frame():
    """Clinical data, one baseline row per subject, keyed by subject. Returns (clin_dedup, clin_os)."""
    clin_out = AllDataCenters.load_clinical(exclude_idh1_mut=False)
    clin_raw = clin_out[[
        "Subject Id", "Sample collection chronology", time_col, event_col,
        "MGMT promoter methylation", "age_at_diagnosis", "Administrative gender", "p53"]].copy()
    clin_raw["_priority"] = clin_raw["Sample collection chronology"].map(
        {"Baseline": 0, "At recurrence": 1}).fillna(2)
    clin_dedup = (clin_raw.sort_values(["Subject Id", "_priority"])
                  .drop_duplicates("Subject Id", keep="first").drop(columns="_priority"))
    clin_dedup["join_key"] = clin_dedup["Subject Id"].astype(str)
    for c in [time_col, event_col, "age_at_diagnosis"]:
        clin_dedup[c] = pd.to_numeric(clin_dedup[c], errors="coerce")

    all_cov_cols = [v["col"] for v in COVARIATES.values()]
    clin_cols    = ["join_key", time_col, event_col] + all_cov_cols
    clin_os      = clin_dedup[clin_cols].dropna(subset=[time_col, event_col]).copy()

    for cov_name, cov_spec in COVARIATES.items():
        clin_os[cov_name] = cov_spec["transform"](clin_os[cov_spec["col"]])
    if cov_names:
        clin_os = clin_os.dropna(subset=cov_names)
    return clin_dedup, clin_os


# Stratification helper

def apply_split(series, mode):
    if mode == "median":
        thr  = series.median()
        arm  = (series >= thr).astype(int)
        keep = pd.Series(True, index=series.index)
    elif mode == "quartile":
        q25, q75 = series.quantile([0.25, 0.75])
        arm  = (series >= q75).astype(int)
        keep = (series >= q75) | (series <= q25)
    elif mode == "max":
        q75  = series.quantile(0.75)
        arm  = (series >= q75).astype(int)
        keep = pd.Series(True, index=series.index)
    else:
        raise ValueError(f"Unknown split mode: {mode}")
    return arm, keep

# RMST helper

def run_rmst(df_in, nh_col, tau, cov_names=cov_names):
    df = df_in[[time_col, event_col, "arm", nh_col] + cov_names].dropna().copy()
    df = df[df[time_col] > 0]
    if len(df) == 0 or df[time_col].max() < tau:
        return None
    n_high = int(df["arm"].sum())
    n_low  = int((df["arm"] == 0).sum())
    if n_high < 10 or n_low < 10:
        return None

    r_df = df[[time_col, event_col, "arm"] + cov_names].copy()
    r_df.columns = ["time", "event", "arm"] + cov_names

    try:
        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv["surv_data"] = ro.conversion.py2rpy(r_df)
        ro.globalenv["tau_val"] = float(tau)

        ro.r("""
        rmst_u <- rmst2(surv_data$time, surv_data$event, surv_data$arm, tau=tau_val)
        u_diff <- rmst_u$unadjusted.result["RMST (arm=1)-(arm=0)", "Est."]
        u_lo   <- rmst_u$unadjusted.result["RMST (arm=1)-(arm=0)", "lower .95"]
        u_hi   <- rmst_u$unadjusted.result["RMST (arm=1)-(arm=0)", "upper .95"]
        u_p    <- rmst_u$unadjusted.result["RMST (arm=1)-(arm=0)", "p"]
        rmst_h <- rmst_u$RMST.arm1$rmst["Est."]
        rmst_l <- rmst_u$RMST.arm0$rmst["Est."]
        """)

        out = {
            "n_high": n_high, "n_low": n_low,
            "rmst_high": float(ro.r("rmst_h")[0]), "rmst_low": float(ro.r("rmst_l")[0]),
            "none_diff": float(ro.r("u_diff")[0]), "none_lo": float(ro.r("u_lo")[0]),
            "none_hi": float(ro.r("u_hi")[0]), "none_p": float(ro.r("u_p")[0]),
        }

        if len(cov_names) > 0:
            cov_parts = []
            for c in cov_names:
                cov_parts.append(f"scale(surv_data${c})[,1]" if c == "age" else f"surv_data${c}")
            cov_r_str = ", ".join(cov_parts)
            ro.r(f"""
            cov_mat <- cbind({cov_r_str})
            rmst_a <- rmst2(surv_data$time, surv_data$event, surv_data$arm,
                            tau=tau_val, covariates=cov_mat)
            a_diff <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "Est."]
            a_lo   <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "lower .95"]
            a_hi   <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "upper .95"]
            a_p    <- rmst_a$adjusted.result["RMST (arm=1)-(arm=0)", "p"]
            """)
            out.update({"active_diff": float(ro.r("a_diff")[0]), "active_lo": float(ro.r("a_lo")[0]),
                        "active_hi": float(ro.r("a_hi")[0]), "active_p": float(ro.r("a_p")[0])})
        else:
            out.update({"active_diff": out["none_diff"], "active_lo": out["none_lo"],
                        "active_hi": out["none_hi"], "active_p": out["none_p"]})
        return out

    except Exception as e:
        print(f"    survRM2 error: {e}")
        return None


def fmt_p(p):
    if np.isnan(p):
        return "n/a"
    if p < 0.0001:
        return f"{p:.2e}"
    return f"{p:.4f}"

def run_sweep(nhood_key, annotation_csv, tau=tau_years, prefix=""):
    """RMST sweep over the neighborhoods of nhood_key. Returns sweep_df, M and fracs."""
    # Neighborhood fractions from the saved per-spot annotations
    sdf = pd.read_csv(annotation_csv, usecols=["sample", nhood_key])
    sdf[nhood_key] = sdf[nhood_key].astype("category")
    counts = sdf.groupby(["sample", nhood_key], observed=False).size().unstack(fill_value=0)
    counts = counts.loc[counts.sum(axis=1) >= min_spots]
    fracs  = counts.div(counts.sum(axis=1), axis=0)
    print(f"Samples with >= {min_spots} spots: {len(fracs)}  |  neighborhoods: {list(fracs.columns)}")

    map_df             = pd.DataFrame({"sample": fracs.index.astype(str)})
    map_df["join_key"] = (map_df["sample"].str.replace(r"_vis$", "", regex=True)
                          .str.replace(r"[a-z]$", "", regex=True))

    clin_os = clinical_frame()[1]
    merge_cols = ["join_key", time_col, event_col] + cov_names
    M = (fracs.reset_index()
             .merge(map_df, on="sample", how="left")
             .merge(clin_os[merge_cols], on="join_key", how="inner"))
    print(f"Merged patients: {M['sample'].nunique()}")

    # Sweep
    all_results = []
    for split_mode in SPLIT_MODES:
        for nh_col in fracs.columns:
            df = M[[time_col, event_col, nh_col] + cov_names].dropna().copy()
            df = df[df[time_col] > 0].copy()
            arm, keep = apply_split(df[nh_col], split_mode)
            df["arm"] = arm
            df = df[keep].copy()
            res = run_rmst(df, nh_col, tau)
            if res:
                res["neighborhood"] = f"{prefix}{nh_col}"
                res["split_mode"]   = split_mode
                all_results.append(res)
            else:
                print(f"  {nh_col}: skipped")

    sweep_df = pd.DataFrame(all_results)

    # Sweep summary with nominal p values and no FDR correction
    print(f"\nSWEEP SUMMARY  |  {nhood_key}  |  covariates: {cov_label}  |  tau={tau}y")
    for mode in SPLIT_MODES:
        sub = sweep_df[sweep_df["split_mode"] == mode].copy()
        if len(sub) == 0:
            continue
        n_sig = (sub["active_p"] < 0.05).sum()
        best_p = sub["active_p"].min()
        print(f"\n  {mode}: {n_sig} sig (p<0.05), best p={best_p:.4f}")
        print(sub[["neighborhood", "n_high", "n_low", "active_diff", "active_p"]]
              .assign(active_diff=lambda d: (d["active_diff"] * 12).round(2))
              .sort_values("active_p").to_string(index=False))
    return sweep_df, M, fracs


def forest_plot(sweep_df, fracs, title, prefix=""):
    """Forest plot for the split named in FOREST_SPLIT."""
    sub  = sweep_df[sweep_df["split_mode"] == FOREST_SPLIT].copy().sort_values("active_p")
    cats = [f"{prefix}{c}" for c in fracs.columns]
    cmap_obj = plt.get_cmap(cmap_name, len(cats))
    cm = {c: to_hex(cmap_obj(i)) for i, c in enumerate(cats)}

    rows = []
    for _, row in sub.iterrows():
        d = row["active_diff"] * 12
        rows.append({"label": str(row["neighborhood"]), "diff": d,
                     "lo": row["active_lo"] * 12, "hi": row["active_hi"] * 12,
                     "p": row["active_p"], "sig": row["active_p"] < FP_SIG_ALPHA,
                     "dir": "pos" if d >= 0 else "neg",
                     "color": cm.get(str(row["neighborhood"]), "#666666")})

    rows_pos = sorted([r for r in rows if r["dir"] == "pos"], key=lambda x: x["diff"], reverse=True)
    rows_neg = sorted([r for r in rows if r["dir"] == "neg"], key=lambda x: x["diff"], reverse=True)
    fp_rows  = rows_pos + rows_neg
    n_rows   = len(fp_rows)
    y_pos    = np.arange(n_rows - 1, -1, -1, dtype=float)

    fig, ax = plt.subplots(figsize=(FP_FIG_W, FP_FIG_H))
    for i, row in enumerate(fp_rows):
        ax.errorbar(x=row["diff"], y=y_pos[i],
                    xerr=[[row["diff"] - row["lo"]], [row["hi"] - row["diff"]]],
                    fmt=FP_MARKER, color=row["color"], markersize=FP_MARKER_SZ,
                    capsize=FP_CAP_SZ, linewidth=FP_LW, elinewidth=FP_ELW, zorder=3)
    ax.axvline(x=0, color="#7F8C8D", linewidth=0.9, linestyle="--", zorder=2)

    all_lo = [r["lo"] for r in fp_rows]
    all_hi = [r["hi"] for r in fp_rows]
    x_pad  = max((max(all_hi) - min(all_lo)) * 0.08, 0.5)
    ax.set_xlim(min(all_lo) - x_pad, max(all_hi) + x_pad)
    ax.set_ylim(-0.7, n_rows - 0.3)
    ax.set_yticks([])
    ax.set_xlabel(f"RMST difference (months, tau={tau_years}y)  |  high vs low neighborhood fraction",
                  fontsize=FP_FS_AXIS)
    ax.set_title(title, fontsize=FP_FS_TITLE, pad=8)
    for sp in ["top", "right", "left"]:
        ax.spines[sp].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.6)
    ax.tick_params(axis="x", labelsize=FP_FS_AXIS, width=0.6, length=3)
    ax.grid(axis="x", linewidth=0.3, alpha=0.4, zorder=0)

    trans = ax.transAxes
    y_lo, y_hi = ax.get_ylim()

    def y_frac(y_data):
        return (y_data - y_lo) / (y_hi - y_lo)

    header_frac = y_frac(n_rows - 0.1)
    ax.text(X_LABEL, header_frac, "Neighborhood", transform=trans, fontsize=FP_FS_HDR,
            fontweight="bold", ha="right", va="bottom")
    ax.text(X_EST, header_frac, "RMST diff (months)", transform=trans, fontsize=FP_FS_HDR,
            fontweight="bold", ha="left", va="bottom")
    ax.text(X_P, header_frac, "adj p", transform=trans, fontsize=FP_FS_HDR,
            fontweight="bold", ha="left", va="bottom")

    for i, row in enumerate(fp_rows):
        yf  = y_frac(y_pos[i])
        col = row["color"] if row["sig"] else FP_COL_NS
        ax.text(X_LABEL, yf, row["label"], transform=trans, fontsize=FP_FS_ROW,
                ha="right", va="center", color="#2C3E50")
        ax.text(X_EST, yf, f"{row['diff']:+.1f}", transform=trans, fontsize=FP_FS_ROW,
                ha="left", va="center", color=col)
        ax.text(X_P, yf, fmt_p(row["p"]) + (" *" if row["sig"] else ""), transform=trans,
                fontsize=FP_FS_ROW, ha="left", va="center", color=col)

    fig.subplots_adjust(left=0.28, right=0.78)
    return fig, sub


def dagger(sweep_df_15, nhood_key):
    """Neighborhoods with a significant RMST association at TAU_CHECK."""
    print(f"{nhood_key}  |  tau={TAU_CHECK}y  |  covariates: {cov_label}")
    for mode in SPLIT_MODES:
        sub = sweep_df_15[sweep_df_15["split_mode"] == mode].copy()
        if len(sub) == 0:
            continue
        sub["diff_mo"] = sub["active_diff"] * 12
        sig = sub[sub["active_p"] < ALPHA].sort_values("active_p")
        print(f"\n  {mode}: {len(sig)} significant (p<{ALPHA})")
        if len(sig):
            for _, r in sig.iterrows():
                direction = "protective" if r["diff_mo"] >= 0 else "hazardous"
                print(f"    {r['neighborhood']:>6}  {r['diff_mo']:+6.1f} mo  "
                      f"p={fmt_p(r['active_p'])}  n={int(r['n_high'])}/{int(r['n_low'])}  {direction}")
        else:
            print("    none")


def describe_fit(M, fracs):
    """Sections without a subject, fitted patients outside the survival cohort, missing covariates."""
    unjoined = sorted(set(fracs.index.astype(str)) - set(M["sample"]))
    n_backup = sum(s.endswith(("_vbu", "_vb2", "_bu2", "_bu3")) for s in unjoined)
    print(f"Samples without a subject: {len(unjoined)} ({n_backup} backup sections)")
    fitted = clinical_frame()[0].set_index("join_key").loc[M["join_key"].unique()]
    outside = sorted(set(fitted.index) - set(cohort.survival_cohort()["patient"]))
    print(f"Fitted patients outside the 260-patient survival cohort: {len(outside)} {outside}")
    for name, spec in COVARIATES.items():
        if name != "age":
            print(f"  {spec['col']}: {int(fitted[spec['col']].isna().sum())} missing, "
                  "counted as the reference level")

## Fig 4h: ECM neighborhoods

In [ ]:
sweep_df, M, fracs = run_sweep("ECM_neighborhood", ECM_CSV, prefix="ECM_")
describe_fit(M, fracs)
fig, ecm_forest = forest_plot(sweep_df, fracs, prefix="ECM_", title=TITLE.format(
    name="ECM", tau=tau_years, split=FOREST_SPLIT, covs=cov_label))
output.save(OUT, "fig4h_ecm_rmst_forest", fig, ecm_forest, panel="Fig 4h")

The † of Fig 4h: neighborhoods significant at τ = 1.5 years in the same model.

In [ ]:
sweep_df_15, _, _ = run_sweep("ECM_neighborhood", ECM_CSV, tau=TAU_CHECK, prefix="ECM_")
dagger(sweep_df_15, "ECM_neighborhood")

ECM_3 and ECM_1 in the text.

In [ ]:
for lab in ("ECM_3", "ECM_1"):
    r = ecm_forest.set_index("neighborhood").loc[lab]
    print(f"{lab}: adjusted {r.active_diff * 12:+.1f} mo [{r.active_lo * 12:.1f}, {r.active_hi * 12:.1f}], "
          f"p = {r.active_p:.4f}; unadjusted {r.none_diff * 12:+.1f} mo, p = {r.none_p:.4f}; n = {r.n_high}/{r.n_low}")

## Supp Fig 4c–d: cell-type neighborhoods (CT_3 and CT_0 in the text)

In [ ]:
ct_sweep_df, ct_M, ct_fracs = run_sweep("celltype_neighborhood", CT_CSV, prefix="CT_")
fig, ct_forest = forest_plot(ct_sweep_df, ct_fracs, prefix="CT_", title=TITLE.format(
    name="Cell type", tau=tau_years, split=FOREST_SPLIT, covs=cov_label))
output.save(OUT, "suppfig4c_ct_rmst_forest", fig, ct_forest, panel="Supp Fig 4c")
ct_sweep_df_15, _, _ = run_sweep("celltype_neighborhood", CT_CSV, tau=TAU_CHECK, prefix="CT_")
dagger(ct_sweep_df_15, "celltype_neighborhood")

CT_0 and CT_3 mutually adjusted: the same model with the other neighborhood's fraction
added as a covariate.

In [ ]:
rows = []
for nh, other in ((3, 0), (0, 3)):
    df = ct_M.rename(columns={other: "other_ct"})[[time_col, event_col, nh, "other_ct"] + cov_names].dropna()
    df["arm"], _ = apply_split(df[nh], FOREST_SPLIT)
    for tau in (tau_years, TAU_CHECK):
        for covs in (cov_names, cov_names + ["other_ct"]):
            r = run_rmst(df, nh, tau, covs)
            rows.append({"neighborhood": f"CT_{nh}", "tau_years": tau,
                         "covariates": " + ".join(covs).replace("other_ct", f"CT_{other}"),
                         "n_high": r["n_high"], "n_low": r["n_low"], "diff_months": r["active_diff"] * 12,
                         "lo": r["active_lo"] * 12, "hi": r["active_hi"] * 12, "p": r["active_p"]})
mutual = pd.DataFrame(rows).round(4)
output.save_table(OUT, "suppfig4c_ct_mutual_adjustment", mutual, index=False)
mutual

In [ ]:
print(output.versions(["numpy", "pandas", "matplotlib", "rpy2"], ["survRM2"]).to_string(index=False))